[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/05_tag_2_3_mlp_dense_hyperparameter.ipynb)

# Tag 2/3 - 05 MLP und Dense-Hyperparameter

Ein Multilayer Perceptron (MLP) ist ein Netz aus mehreren vollständig verbundenen (`Dense`) Layern. Hier wird das Modell bewusst **nicht** mit einer Schleife gebaut, sondern Schritt für Schritt, damit die Architektur sichtbar bleibt.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


## Datensatz

Wir nutzen `make_moons`: zwei Klassen, die mit einer geraden Linie nicht sauber getrennt werden können. Das passt gut zum MLP, weil Hidden Layer nichtlineare Grenzen lernen können.


In [ ]:
X, y = make_moons(n_samples=800, noise=0.22, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=20, edgecolor="black", alpha=0.75)
plt.title("Two-Moons-Datensatz")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


## Wichtige `Dense`-Parameter

| Parameter | Bedeutung |
| --- | --- |
| `units` | Anzahl der Neuronen im Layer |
| `activation` | Aktivierungsfunktion nach der linearen Transformation |
| `use_bias` | Bias-Term ein- oder ausschalten |
| `kernel_initializer` | Startwerte der Gewichte |
| `kernel_regularizer` | L1/L2-Strafe auf Gewichte |

Im Code sind die Parameter, die wir gerade betrachten, direkt kommentiert.


In [ ]:
model = keras.Sequential(name="mlp_sequential_demo")

model.add(keras.Input(shape=(2,), name="features"))

# Parameter im Fokus: units=16 bedeutet 16 Neuronen im ersten Hidden Layer.
# Parameter im Fokus: activation="relu" macht das Netz nichtlinear.
model.add(layers.Dense(units=16, activation="relu", name="hidden_1"))

# Zweiter Hidden Layer: Das Modell kann dadurch komplexere Zwischenmerkmale bilden.
model.add(layers.Dense(units=16, activation="relu", name="hidden_2"))

# Output Layer: 1 Neuron + Sigmoid für binäre Klassifikation.
model.add(layers.Dense(units=1, activation="sigmoid", name="output"))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
def draw_mlp_diagram():
    fig, ax = plt.subplots(figsize=(8, 2.4))
    ax.axis("off")
    labels = ["Input\n2", "hidden_1\n16 ReLU", "hidden_2\n16 ReLU", "output\n1 Sigmoid"]
    xs = [0.05, 0.31, 0.57, 0.80]
    widths = [0.18, 0.20, 0.20, 0.18]
    for x, width, label in zip(xs, widths, labels):
        ax.add_patch(plt.Rectangle((x, 0.34), width, 0.36, fill=False, linewidth=2))
        ax.text(x + width / 2, 0.52, label, ha="center", va="center")
    for x1, x2 in [(0.23, 0.31), (0.51, 0.57), (0.77, 0.80)]:
        ax.annotate("", xy=(x2, 0.52), xytext=(x1, 0.52), arrowprops=dict(arrowstyle="->", linewidth=2))
    ax.set_title("MLP als sequentielle Layer-Kette")
    plt.show()


draw_mlp_diagram()


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.25,
    epochs=55,
    batch_size=32,
    verbose=0,
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.3f}")
print(f"Test Accuracy: {test_accuracy:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 180),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
proba = model.predict(grid, verbose=0).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
plt.contour(xx, yy, proba, levels=[0.5], colors="black")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=22, edgecolor="black")
plt.title("Gelerntes Entscheidungsgebiet")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()
